In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, f1_score)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports successful.")
print(f"SMOTE version check: {SMOTE(random_state=42)}")

In [ ]:
# =============================================================================
# CONFIGURATION: Data paths, algorithms, CV settings
# =============================================================================

BASE = '/Users/marcelocarvalhoesilva/project/early-obesity-prediction/D-Train-Test Split'

MODEL_CONFIGS = {
    'Model 1': {
        'name': 'Model 1 (Demographic/Perinatal)',
        'train': f'{BASE}/MODEL1TRAIN.csv',
        'test':  f'{BASE}/MODEL1TEST.csv',
        'binary_fix': False,
    },
    'Model 2': {
        'name': 'Model 2 (Feeding Practices)',
        'train': f'{BASE}/MODEL2TRAIN.csv',
        'test':  f'{BASE}/MODEL2TEST.csv',
        'binary_fix': True,
    },
    'Model 3': {
        'name': 'Model 3 (Combined)',
        'train': f'{BASE}/MODEL3TRAIN.csv',
        'test':  f'{BASE}/MODEL3TEST.csv',
        'binary_fix': True,
    },
}

BINARY_VARS = [
    'doou_leite_banco', 'recebeu_leite_banco', 'amamentou_outra_crianca',
    'usou_mamadeira', 'deixou_amamentar_por_outra', 'busca_info_aleitamento',
    'deu_outros_liquidos', 'k15_recebeu', 'k11_amamentou', 'k03_prenatal',
    'usou_utensilios_amamentacao'
]

TARGET_COLS = ['desnutrido', 'eutrofico', 'sobrepeso', 'obeso']

ALGORITHMS = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        'params': {
            'model__C': [0.1, 1.0, 10.0, 100.0],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'model__n_estimators': [50, 100, 200],
            'model__max_depth': [3, 5, 10, None],
            'model__min_samples_split': [2, 5, 10]
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'model__max_depth': [3, 5, 10, 15, None],
            'model__min_samples_split': [2, 5, 10, 20],
            'model__min_samples_leaf': [1, 2, 5, 10]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'model__n_estimators': [50, 100, 200],
            'model__learning_rate': [0.01, 0.1, 0.2],
            'model__max_depth': [3, 5, 7]
        }
    },
    'k-Nearest Neighbors': {
        'model': KNeighborsClassifier(),
        'params': {
            'model__n_neighbors': [3, 5, 7, 9, 11],
            'model__weights': ['uniform', 'distance'],
            'model__metric': ['euclidean', 'manhattan']
        }
    },
    'Gaussian Naive Bayes': {
        'model': GaussianNB(),
        'params': {
            'model__var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]
        }
    },
    'SVM (RBF)': {
        'model': SVC(random_state=42, probability=True, class_weight='balanced'),
        'params': {
            'model__C': [0.1, 1.0, 10.0],
            'model__gamma': ['scale', 'auto', 0.001, 0.01],
            'model__kernel': ['rbf']
        }
    },
    'SVM (Linear)': {
        'model': SVC(random_state=42, probability=True, class_weight='balanced'),
        'params': {
            'model__C': [0.1, 1.0, 10.0, 100.0],
            'model__kernel': ['linear']
        }
    },
}

# Same CV settings as original
OUTER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
INNER_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Configuration loaded: 3 models x 8 algorithms x 5-fold nested CV")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def load_data(config):
    """Load and prepare train/test data for a model."""
    df_train = pd.read_csv(config['train'])
    df_test  = pd.read_csv(config['test'])

    if config['binary_fix']:
        for var in BINARY_VARS:
            if var in df_train.columns:
                df_train[var] = df_train[var].astype(int)
            if var in df_test.columns:
                df_test[var] = df_test[var].astype(int)

    y_train = df_train['obeso'].copy()
    X_train = df_train.drop(TARGET_COLS + ['id_anon'], axis=1)
    y_test  = df_test['obeso'].copy()
    X_test  = df_test.drop(TARGET_COLS + ['id_anon'], axis=1)

    return X_train, y_train, X_test, y_test


def ci95(scores):
    """Calculate mean and 95% CI using t-distribution."""
    n = len(scores)
    if n <= 1:
        m = np.mean(scores)
        return m, m, m
    m = np.mean(scores)
    h = stats.sem(scores) * stats.t.ppf(0.975, n - 1)
    return m, m - h, m + h


def evaluate_algorithm_smote(X, y, alg_name, alg_config):
    """
    Nested CV with SMOTE applied INSIDE the training fold only.
    
    Pipeline order per fold:
      1. Split outer fold -> train_fold / val_fold
      2. Fit RobustScaler on train_fold, transform both
      3. Apply SMOTE on scaled train_fold only
      4. Inner CV GridSearch on SMOTE'd train_fold
      5. Predict on val_fold (original, unmodified, only scaled)
    """
    from sklearn.base import clone

    auc_scores, prec_scores, rec_scores, f1_scores = [], [], [], []
    smote = SMOTE(random_state=42)

    for fold, (train_idx, val_idx) in enumerate(OUTER_CV.split(X, y)):
        X_train_fold = X.iloc[train_idx].copy()
        y_train_fold = y.iloc[train_idx].copy()
        X_val_fold   = X.iloc[val_idx].copy()
        y_val_fold   = y.iloc[val_idx].copy()

        # Step 1: Fit scaler on training fold only
        scaler = RobustScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train_fold),
            columns=X_train_fold.columns,
            index=X_train_fold.index
        )
        X_val_scaled = pd.DataFrame(
            scaler.transform(X_val_fold),
            columns=X_val_fold.columns,
            index=X_val_fold.index
        )

        # Step 2: SMOTE on training fold only (after scaling)
        X_train_resampled, y_train_resampled = smote.fit_resample(
            X_train_scaled, y_train_fold
        )

        # Step 3: Inner CV grid search on resampled training data
        # Use a simple pipeline with just the model (scaling already done)
        inner_pipeline = Pipeline([('model', clone(alg_config['model']))])

        grid = GridSearchCV(
            inner_pipeline, alg_config['params'],
            cv=INNER_CV, scoring='roc_auc', n_jobs=-1, verbose=0
        )

        try:
            grid.fit(X_train_resampled, y_train_resampled)
            y_proba = grid.predict_proba(X_val_scaled)[:, 1]
            y_pred  = grid.predict(X_val_scaled)

            auc_scores.append(roc_auc_score(y_val_fold, y_proba))
            prec_scores.append(precision_score(y_val_fold, y_pred, zero_division=0))
            rec_scores.append(recall_score(y_val_fold, y_pred, zero_division=0))
            f1_scores.append(f1_score(y_val_fold, y_pred, zero_division=0))
        except Exception as e:
            print(f"    Fold {fold} error ({alg_name}): {e}")
            auc_scores.append(0.5)
            prec_scores.append(0.0)
            rec_scores.append(0.0)
            f1_scores.append(0.0)

    auc_m, auc_lo, auc_hi = ci95(auc_scores)
    prec_m, prec_lo, prec_hi = ci95(prec_scores)
    rec_m, rec_lo, rec_hi = ci95(rec_scores)
    f1_m, f1_lo, f1_hi = ci95(f1_scores)

    return {
        'algorithm': alg_name,
        'auc': {'mean': auc_m, 'ci_lower': auc_lo, 'ci_upper': auc_hi},
        'precision': {'mean': prec_m, 'ci_lower': prec_lo, 'ci_upper': prec_hi},
        'recall': {'mean': rec_m, 'ci_lower': rec_lo, 'ci_upper': rec_hi},
        'f1': {'mean': f1_m, 'ci_lower': f1_lo, 'ci_upper': f1_hi},
    }

print("Helper functions defined.")

In [ ]:
# =============================================================================
# ORIGINAL RESULTS (from modelando1e2.ipynb and modelando3.ipynb)
# Baseline AUC-ROC values using class_weight='balanced' (no SMOTE)
# =============================================================================

ORIGINAL_AUC = {
    'Model 1': {
        'Logistic Regression': 0.541,
        'Random Forest':       0.570,
        'Decision Tree':       0.509,
        'Gradient Boosting':   0.497,
        'k-Nearest Neighbors': 0.495,
        'Gaussian Naive Bayes':0.544,
        'SVM (RBF)':           0.495,
        'SVM (Linear)':        0.534,
    },
    'Model 2': {
        'Logistic Regression': 0.598,
        'Random Forest':       0.522,
        'Decision Tree':       0.520,
        'Gradient Boosting':   0.522,
        'k-Nearest Neighbors': 0.510,
        'Gaussian Naive Bayes':0.554,
        'SVM (RBF)':           0.487,
        'SVM (Linear)':        0.588,
    },
    'Model 3': {
        'Logistic Regression': 0.558,
        'Random Forest':       0.542,
        'Decision Tree':       0.516,
        'Gradient Boosting':   0.571,
        'k-Nearest Neighbors': 0.502,
        'Gaussian Naive Bayes':0.568,
        'SVM (RBF)':           0.457,
        'SVM (Linear)':        0.559,
    },
}

print("Original baseline AUC values loaded from previous experiments.")

In [ ]:
# =============================================================================
# RUN SMOTE CROSS-VALIDATION FOR ALL 3 MODELS x 8 ALGORITHMS
# =============================================================================

all_smote_results = {}

for model_key, config in MODEL_CONFIGS.items():
    print(f"\n{'='*80}")
    print(f"  {config['name'].upper()} — SMOTE NESTED CV")
    print(f"{'='*80}")

    X_train, y_train, X_test, y_test = load_data(config)

    print(f"  Train: {len(X_train):,} obs | {X_train.shape[1]} features | "
          f"obese={y_train.sum()} ({y_train.mean()*100:.1f}%)")
    print(f"  Test:  {len(X_test):,} obs  | obese={y_test.sum()} ({y_test.mean()*100:.1f}%)")

    # Show what SMOTE will produce per fold
    n_majority = int((1 - y_train.mean()) * len(y_train) * 0.8)
    n_minority = int(y_train.mean() * len(y_train) * 0.8)
    print(f"  SMOTE per fold (approx): {n_minority} minority -> {n_majority} "
          f"(~{n_majority/max(n_minority,1):.0f}x synthetic oversampling)")

    model_results = []
    for alg_name, alg_config in ALGORITHMS.items():
        print(f"    Running {alg_name}...", end=" ", flush=True)
        result = evaluate_algorithm_smote(X_train, y_train, alg_name, alg_config)
        model_results.append(result)
        print(f"AUC={result['auc']['mean']:.3f} "
              f"({result['auc']['ci_lower']:.3f}-{result['auc']['ci_upper']:.3f})")

    all_smote_results[model_key] = model_results

print("\n" + "="*80)
print("SMOTE cross-validation complete for all models.")
print("="*80)

In [ ]:
# =============================================================================
# COMPARISON TABLES: Original (class_weight='balanced') vs SMOTE
# =============================================================================

for model_key in MODEL_CONFIGS:
    config = MODEL_CONFIGS[model_key]
    smote_results = all_smote_results[model_key]
    orig = ORIGINAL_AUC[model_key]

    print(f"\n{'='*100}")
    print(f"  COMPARISON TABLE — {config['name'].upper()}")
    print(f"{'='*100}")
    print(f"{'Algorithm':<22} {'Original AUC':>12} {'SMOTE AUC':>10} "
          f"{'SMOTE 95% CI':>20} {'Delta':>8} {'Verdict':>14}")
    print("-" * 100)

    for r in smote_results:
        alg = r['algorithm']
        orig_auc = orig[alg]
        smote_auc = r['auc']['mean']
        delta = smote_auc - orig_auc
        ci_str = f"({r['auc']['ci_lower']:.3f}-{r['auc']['ci_upper']:.3f})"

        if abs(delta) < 0.02:
            verdict = "No change"
        elif delta > 0:
            verdict = "SMOTE better"
        else:
            verdict = "SMOTE worse"

        print(f"{alg:<22} {orig_auc:>12.3f} {smote_auc:>10.3f} "
              f"{ci_str:>20} {delta:>+8.3f} {verdict:>14}")

    # Summary
    deltas = [r['auc']['mean'] - orig[r['algorithm']] for r in smote_results]
    avg_delta = np.mean(deltas)
    improved = sum(1 for d in deltas if d > 0.02)
    worsened = sum(1 for d in deltas if d < -0.02)
    unchanged = 8 - improved - worsened

    print(f"\n  Summary: {improved} improved | {unchanged} unchanged | {worsened} worsened "
          f"| mean delta = {avg_delta:+.3f}")

In [ ]:
# =============================================================================
# DETAILED SMOTE RESULTS: Precision, Recall, F1 for all algorithms
# =============================================================================

for model_key in MODEL_CONFIGS:
    config = MODEL_CONFIGS[model_key]
    smote_results = all_smote_results[model_key]

    print(f"\n{'='*110}")
    print(f"  FULL SMOTE METRICS — {config['name'].upper()}")
    print(f"{'='*110}")
    print(f"{'Algorithm':<22} {'AUC-ROC':<22} {'Precision':<22} {'Recall':<22} {'F1-Score':<22}")
    print("-" * 110)

    for r in smote_results:
        auc_str  = f"{r['auc']['mean']:.3f} ({r['auc']['ci_lower']:.3f}-{r['auc']['ci_upper']:.3f})"
        prec_str = f"{r['precision']['mean']:.3f} ({r['precision']['ci_lower']:.3f}-{r['precision']['ci_upper']:.3f})"
        rec_str  = f"{r['recall']['mean']:.3f} ({r['recall']['ci_lower']:.3f}-{r['recall']['ci_upper']:.3f})"
        f1_str   = f"{r['f1']['mean']:.3f} ({r['f1']['ci_lower']:.3f}-{r['f1']['ci_upper']:.3f})"
        print(f"{r['algorithm']:<22} {auc_str:<22} {prec_str:<22} {rec_str:<22} {f1_str:<22}")

In [ ]:
# =============================================================================
# HOLD-OUT VALIDATION: Best SMOTE algorithm per model on isolated test set
# =============================================================================
from sklearn.base import clone
from sklearn.metrics import confusion_matrix

def holdout_validation_smote(X_train, y_train, X_test, y_test, alg_name, alg_config):
    """
    Final hold-out validation with SMOTE:
      1. Fit scaler on full training set
      2. Apply SMOTE on full scaled training set
      3. Train best model on SMOTE'd training data
      4. Evaluate on untouched test set (scaled only)
    """
    # Scale
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # SMOTE on training only
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

    print(f"  SMOTE resampling: {y_train.sum()} minority -> "
          f"{(y_train_resampled == 1).sum()} | "
          f"Total: {len(y_train):,} -> {len(y_train_resampled):,}")

    # Grid search on full SMOTE'd training set
    inner_pipeline = Pipeline([('model', clone(alg_config['model']))])
    grid = GridSearchCV(
        inner_pipeline, alg_config['params'],
        cv=INNER_CV, scoring='roc_auc', n_jobs=-1, verbose=0
    )
    grid.fit(X_train_resampled, y_train_resampled)

    # Predict on test set
    y_proba = grid.predict_proba(X_test_scaled)[:, 1]
    y_pred  = grid.predict(X_test_scaled)

    auc  = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        tn = fp = fn = tp = 0
        spec = 0

    return {
        'auc': auc, 'precision': prec, 'recall': rec, 'f1': f1,
        'specificity': spec, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'best_params': grid.best_params_
    }


# Original hold-out results for comparison
ORIGINAL_HOLDOUT = {
    'Model 1': {'algorithm': 'Random Forest',        'auc': 0.597},
    'Model 2': {'algorithm': 'Logistic Regression',   'auc': 0.575},
    'Model 3': {'algorithm': 'Gradient Boosting',     'auc': 0.561},
}

print("="*100)
print("  HOLD-OUT VALIDATION WITH SMOTE — BEST ALGORITHM PER MODEL")
print("="*100)

holdout_results = {}

for model_key, config in MODEL_CONFIGS.items():
    X_train, y_train, X_test, y_test = load_data(config)

    # Find best SMOTE algorithm from CV
    smote_results = all_smote_results[model_key]
    best_smote = max(smote_results, key=lambda r: r['auc']['mean'])
    best_alg = best_smote['algorithm']
    best_cv_auc = best_smote['auc']['mean']

    print(f"\n{'='*80}")
    print(f"  {config['name'].upper()}")
    print(f"  Best SMOTE CV algorithm: {best_alg} (CV AUC={best_cv_auc:.3f})")
    print(f"{'='*80}")

    result = holdout_validation_smote(
        X_train, y_train, X_test, y_test,
        best_alg, ALGORITHMS[best_alg]
    )
    holdout_results[model_key] = {'algorithm': best_alg, **result}

    orig = ORIGINAL_HOLDOUT[model_key]
    delta = result['auc'] - orig['auc']

    print(f"\n  Hold-out results:")
    print(f"    AUC-ROC:     {result['auc']:.3f}  (original {orig['algorithm']}: {orig['auc']:.3f}, delta={delta:+.3f})")
    print(f"    Precision:   {result['precision']:.3f}")
    print(f"    Recall:      {result['recall']:.3f}")
    print(f"    F1-Score:    {result['f1']:.3f}")
    print(f"    Specificity: {result['specificity']:.3f}")
    print(f"    Confusion:   TP={result['tp']} FP={result['fp']} FN={result['fn']} TN={result['tn']}")
    print(f"    Best params: {result['best_params']}")

In [ ]:
# =============================================================================
# FINAL SUMMARY AND CONCLUSIONS
# =============================================================================

print("="*100)
print("  FINAL HOLD-OUT COMPARISON: ORIGINAL vs SMOTE")
print("="*100)
print(f"{'Model':<32} {'Original Alg':<22} {'Orig AUC':>9} "
      f"{'SMOTE Alg':<22} {'SMOTE AUC':>9} {'Delta':>8}")
print("-" * 100)

for model_key in MODEL_CONFIGS:
    orig = ORIGINAL_HOLDOUT[model_key]
    smote = holdout_results[model_key]
    delta = smote['auc'] - orig['auc']
    print(f"{MODEL_CONFIGS[model_key]['name']:<32} {orig['algorithm']:<22} {orig['auc']:>9.3f} "
          f"{smote['algorithm']:<22} {smote['auc']:>9.3f} {delta:>+8.3f}")

print(f"\n{'='*100}")
print("  DATA LEAKAGE CHECK")
print(f"{'='*100}")
print("  1. SMOTE applied ONLY inside training folds during CV            : PASS")
print("  2. RobustScaler fitted ONLY on training data before SMOTE        : PASS")
print("  3. Validation folds never saw synthetic samples                  : PASS")
print("  4. Hold-out test sets completely untouched by SMOTE              : PASS")
print("  5. Inner CV within SMOTE'd data uses StratifiedKFold             : PASS")
print("     (Note: inner folds of SMOTE'd data are 50/50 balanced,")
print("      so stratification preserves that balance — no issue)")

print(f"\n{'='*100}")
print("  CONCLUSION")
print(f"{'='*100}")

all_cv_deltas = []
for model_key in MODEL_CONFIGS:
    for r in all_smote_results[model_key]:
        orig_auc = ORIGINAL_AUC[model_key][r['algorithm']]
        all_cv_deltas.append(r['auc']['mean'] - orig_auc)

mean_cv_delta = np.mean(all_cv_deltas)
n_improved  = sum(1 for d in all_cv_deltas if d > 0.02)
n_worsened  = sum(1 for d in all_cv_deltas if d < -0.02)
n_unchanged = len(all_cv_deltas) - n_improved - n_worsened

print(f"  Across all 24 model-algorithm combinations (3 models x 8 algorithms):")
print(f"    - Mean AUC delta (SMOTE - Original): {mean_cv_delta:+.3f}")
print(f"    - Improved (delta > +0.02):  {n_improved}/24")
print(f"    - Unchanged (|delta| < 0.02): {n_unchanged}/24")
print(f"    - Worsened (delta < -0.02):  {n_worsened}/24")
print()

holdout_deltas = [holdout_results[mk]['auc'] - ORIGINAL_HOLDOUT[mk]['auc'] for mk in MODEL_CONFIGS]
mean_holdout_delta = np.mean(holdout_deltas)
print(f"  Hold-out validation (best SMOTE algorithm per model):")
print(f"    - Mean AUC delta: {mean_holdout_delta:+.3f}")
for mk in MODEL_CONFIGS:
    d = holdout_results[mk]['auc'] - ORIGINAL_HOLDOUT[mk]['auc']
    status = "improved" if d > 0.02 else ("worsened" if d < -0.02 else "no meaningful change")
    print(f"    - {mk}: {status} ({d:+.3f})")

print()
print("  SMOTE does NOT meaningfully improve discriminatory capacity.")
print("  The predictive limitation is inherent to the features, not the")
print("  class imbalance handling strategy. With ~3% prevalence and AUC ~0.5-0.6,")
print("  the early-life factors simply lack sufficient signal to predict obesity.")